In [1]:
"""
서울시 따릉이
- 전체 이용량 vs 외국인 이용량 비교
- 2024 / 2025 비교
- 외국인 이용 비율 Bar Graph

필요:
pip install pandas matplotlib seaborn
"""
import plotly.express as px
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# ======================================================
# 1. 한글 폰트 설정
# ======================================================

def set_korean_font():

    for font in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:

        if font in {f.name for f in fm.fontManager.ttflist}:
            plt.rcParams["font.family"] = font
            break

    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()


# ======================================================
# 2. CSV 로드 함수
# ======================================================

def load_csv(path):

    for enc in ["utf-8-sig", "euc-kr", "cp949"]:

        try:
            return pd.read_csv(path, encoding=enc)

        except:
            continue

    raise Exception(f"로드 실패: {path}")


# ======================================================
# 3. 파일 경로
# ======================================================

# ---------- 전체 이용량 ----------

ALL_24_1 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 이용정보(월별)_24.1-6.csv"
ALL_24_2 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 이용정보(월별)_24.7-12.csv"

ALL_25_1 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 이용정보(월별)_25.1-6.csv"
ALL_25_2 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 이용정보(월별)_25.7-12.csv"


# ---------- 외국인 이용량 ----------

FOREIGN_24_1 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 외국인 대여정보(월별)_24.1-6.csv"
FOREIGN_24_2 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 외국인 대여정보(월별)_24.7-12.csv"

FOREIGN_25_1 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 외국인 대여정보(일별)_25.1-6.csv"
FOREIGN_25_2 = r"C:\Users\USER\OneDrive\바탕 화면\시각화 공모전 데이터\서울특별시 공공자전거 외국인 대여정보(일별)_25.7-12.csv"


# ======================================================
# 4. 데이터 로드
# ======================================================

# 전체 이용량
all_24 = pd.concat([
    load_csv(ALL_24_1),
    load_csv(ALL_24_2)
], ignore_index=True)

all_25 = pd.concat([
    load_csv(ALL_25_1),
    load_csv(ALL_25_2)
], ignore_index=True)

# 외국인 이용량
foreign_24 = pd.concat([
    load_csv(FOREIGN_24_1),
    load_csv(FOREIGN_24_2)
], ignore_index=True)

foreign_25 = pd.concat([
    load_csv(FOREIGN_25_1),
    load_csv(FOREIGN_25_2)
], ignore_index=True)


# ======================================================
# 5. 컬럼 정리
# ======================================================

for df in [all_24, all_25, foreign_24, foreign_25]:

    df.columns = (
        df.columns
        .str.strip()
    )


# ======================================================
# 6. 총 대여량 계산
# ======================================================

# 전체 이용량
total_24 = all_24['이용건수'].sum()
total_25 = all_25['이용건수'].sum()

# 외국인 이용량
foreign_total_24 = foreign_24['대여건수'].sum()
foreign_total_25 = foreign_25['대여건수'].sum()


# ======================================================
# 7. 외국인 비율 계산
# ======================================================

foreign_ratio_24 = (
    foreign_total_24 / total_24
) * 100

foreign_ratio_25 = (
    foreign_total_25 / total_25
) * 100


# ======================================================
# 8. 결과 테이블
# ======================================================

result_df = pd.DataFrame({

    '연도': ['2024', '2025'],

    '전체대여량': [
        total_24,
        total_25
    ],

    '외국인대여량': [
        foreign_total_24,
        foreign_total_25
    ],

    '외국인비율(%)': [
        foreign_ratio_24,
        foreign_ratio_25
    ]
})

print(result_df)


# ======================================================
# 9. 외국인 비율 바 그래프
# ======================================================
fig = px.bar(

    result_df,

    x='연도',
    y='외국인비율(%)',

    text='외국인비율(%)',

    color='연도',

    title='서울시 따릉이 외국인 이용 비율 비교'
)

fig.update_traces(

    texttemplate='%{text:.2f}%',

    textposition='outside',

    marker_line_width=1.5
)

fig.update_layout(

    template='plotly_white',

    title_font_size=24,

    xaxis_title='연도',
    yaxis_title='외국인 이용 비율 (%)',

    font=dict(size=15),

    showlegend=False,

    height=600,
    width=900
)
# 이미지 저장
fig.write_image(
    "foreign_bike_ratio.png",
    width=1200,
    height=800,
    scale=2
)

fig.show()

# ======================================================
# 10. 전체 vs 외국인 이용량 비교
# ======================================================

compare_df = pd.DataFrame({

    '연도': [
        '2024', '2024',
        '2025', '2025'
    ],

    '구분': [
        '전체',
        '외국인',
        '전체',
        '외국인'
    ],

    '대여량': [
        total_24,
        foreign_total_24,
        total_25,
        foreign_total_25
    ]
})
fig = px.bar(

    compare_df,

    x='연도',
    y='대여량',

    color='구분',

    barmode='group',

    text='대여량',

    title='서울시 따릉이 전체 vs 외국인 대여량 비교 (로그 스케일)',

    color_discrete_map={

        '전체': '#4C78A8',
        '외국인': '#F58518'
    }
)

fig.update_traces(

    texttemplate='%{text:,}',

    textposition='outside'
)

# 핵심: 로그 스케일
fig.update_yaxes(
    type='log'
)

fig.update_layout(

    template='plotly_white',

    title_font_size=24,

    xaxis_title='연도',
    yaxis_title='총 대여량 (log scale)',

    font=dict(size=15),

    height=650,
    width=1000
)
fig.write_image(
    "foreign_vs_total_logscale.png",
    width=1400,
    height=900,
    scale=2
)
fig.show()

# ======================================================
# 11. 외국인 이용 증가율
# ======================================================

growth = (
    (foreign_total_25 - foreign_total_24)
    / foreign_total_24
) * 100

growth_df = pd.DataFrame({

    '구분': ['외국인 이용 증가율'],

    '증가율': [growth]
})

fig = px.bar(

    growth_df,

    x='구분',
    y='증가율',

    text='증가율',

    color='증가율',

    title='2024 → 2025 외국인 따릉이 이용 증가율'
)

fig.update_traces(

    texttemplate='%{text:.1f}%',

    textposition='outside'
)

fig.update_layout(

    template='plotly_white',

    title_font_size=24,

    xaxis_title='',
    yaxis_title='증가율 (%)',

    showlegend=False,

    font=dict(size=15),

    height=600,
    width=800
)
fig.show()


     연도     전체대여량  외국인대여량  외국인비율(%)
0  2024  43849559   71077  0.162093
1  2025  37373054   70447  0.188497
